# 🔵 Interagindo com Cassandra usando Python

Este notebook é um guia didático e prático que demonstra como interagir com o **Apache Cassandra** (banco de dados NoSQL do tipo **Família de Colunas** ou **Wide Column Store**) utilizando a linguagem Python e a biblioteca `cassandra-driver`.

## 🛠️ O que é o Cassandra?
Desenvolvido originalmente pelo Facebook, o Apache Cassandra foi projetado para gerenciar **grandes volumes de dados distribuídos** em vários servidores. Ele oferece alta disponibilidade sem pontos únicos de falha. Ao contrário de bancos de dados relacionais e do MongoDB, no Cassandra, a modelagem de dados deve ser orientada estritamente pelas **consultas** que sua aplicação fará (Query-Driven Modeling).

### Resumo Conceitual

| Propriedade | Detalhes |
|---|---|
| **Paradigma** | Família de Colunas (Wide Column Store) |
| **Linguagem de Consulta** | CQL — Cassandra Query Language (similar ao SQL, mas com restrições) |
| **Armazenamento** | Dados distribuídos em partições baseadas em hash da Partition Key |
| **Quando usar** | Séries temporais (IoT, logs), aplicações de altíssima escala de escrita, dados geográficos distribuídos, alta disponibilidade (99.999%) |
| **Quando NÃO usar** | Consultas ad-hoc complexas, dados com muitas relações, aplicações que precisam de JOINs, poucos dados (< 100GB) |

### Terminologia: SQL vs Cassandra

| SQL (Relacional) | Cassandra |
|---|---|
| Banco de Dados | **Keyspace** |
| Tabela | Tabela |
| Linha | **Partição** (pode conter múltiplas linhas internas) |
| Primary Key | **Partition Key + Clustering Key** |
| Índice Secundário | Índice Secundário (menos eficiente) |

### Detalhes da Conexão Local (Docker Compose):
- **Host:** `localhost`
- **Porta:** `9042`
- **Autenticação:** Nenhuma (configuração padrão local)

> **⚠️ Importante:** O Cassandra pode demorar entre **1 e 2 minutos** para inicializar completamente no Docker. Certifique-se de que ele já esteja ativo antes de executar o notebook.

## 📋 Pré-requisitos

Antes de executar este notebook, certifique-se de que:

1. O **Docker** está instalado e em execução na sua máquina.
2. Os containers do projeto foram iniciados com `make up` ou `docker compose up -d`.
3. O container `cassandra` está rodando **e já terminou de inicializar** (verifique com `docker compose ps` — o status deve ser `healthy` ou `Up`).

> **💡 Dica:** Se a conexão falhar com `NoHostAvailable`, aguarde mais 1-2 minutos e tente novamente. O Cassandra é o banco mais lento para inicializar dentre os 4 deste projeto.

## 2. Conectando ao Cluster
No Cassandra, conectamo-nos a um **cluster** composto de um ou mais nós. O driver gerencia a descoberta e balanceamento de conexões automaticamente.

> **💡 Conceito-Chave:** Diferente do Redis e MongoDB, onde nos conectamos a um servidor, no Cassandra nos conectamos a um **cluster** de nós. Passamos uma lista de `contact_points` (endereços iniciais) e o driver descobre automaticamente os demais nós da rede.

**Saída esperada:**
```
✅ Conectado ao cluster: 'Test Cluster' | Versão do Cassandra: 5.0.x
```

In [ ]:
from cassandra.cluster import Cluster

try:
    # Definir os nós de contato (contact points) e a porta
    # Em produção, listaria-se múltiplos IPs: ['10.0.0.1', '10.0.0.2', '10.0.0.3']
    cluster = Cluster(['localhost'], port=9042)
    
    # Estabelecer a sessão de comunicação
    # A sessão é thread-safe e pode ser reutilizada em toda a aplicação
    session = cluster.connect()
    
    # Executar uma query simples de sistema para verificar a conexão
    resultado = session.execute("SELECT cluster_name, release_version FROM system.local")
    for linha in resultado:
        print(f"✅ Conectado ao cluster: '{linha.cluster_name}' | Versão do Cassandra: {linha.release_version}")
        
except Exception as e:
    print(f"❌ Erro ao conectar ao Cassandra: {e}")
    print("Dica: Aguarde mais um pouco se o container do Cassandra tiver acabado de iniciar.")

---
## 3. Criando um Keyspace
No Cassandra, um **Keyspace** é equivalente a um banco de dados nos sistemas tradicionais. Ele define o escopo físico da **replicação dos dados** — ou seja, quantas cópias dos dados serão mantidas e em quais datacenters.

Utilizaremos:
- **`SimpleStrategy`:** Estratégia de replicação para um único datacenter (ideal para desenvolvimento).
- **`replication_factor: 1`:** Apenas uma cópia dos dados (pois temos um único nó local).

> **💡 Em produção:** Usaríamos `NetworkTopologyStrategy` com `replication_factor: 3` (3 cópias em nós diferentes), garantindo que, mesmo se 2 servidores caírem, os dados continuem disponíveis.

**Saída esperada:**
```
🏢 Keyspace 'escola' criado ou já existente.
🎯 Sessão apontando para o Keyspace 'escola'.
```

In [ ]:
# Criar Keyspace se não existir
# IF NOT EXISTS evita erro caso o keyspace já tenha sido criado anteriormente
query_keyspace = """
CREATE KEYSPACE IF NOT EXISTS escola 
WITH replication = {
    'class': 'SimpleStrategy', 
    'replication_factor': 1
};
"""
session.execute(query_keyspace)
print("🏢 Keyspace 'escola' criado ou já existente.")

# Mudar para o contexto do keyspace criado
# Equivale ao "USE escola;" no CQL
session.set_keyspace('escola')
print("🎯 Sessão apontando para o Keyspace 'escola'.")

---
## 4. Criando Tabelas e a Estrutura da Chave Primária
No Cassandra, a **chave primária** (`PRIMARY KEY`) é o conceito mais importante da modelagem de dados. Ela é dividida em duas partes:

```
PRIMARY KEY ((partition_key), clustering_key)
              ▲                 ▲
              │                 │
    Define EM QUAL NÓ      Define a ORDENAÇÃO
    o dado é gravado       dentro do nó
```

1. **Partition Key (Chave de Partição):** Define em qual nó do cluster o dado físico será gravado. O Cassandra aplica uma função hash nesse campo para distribuir os registros uniformemente pelos nós.
2. **Clustering Key (Chave de Agrupamento):** Define a **ordenação física** dos dados dentro da partição.

Nesse exemplo, criaremos uma tabela `estudantes` onde a Primary Key é `((curso), id)`: 
- `curso` é a **Partition Key** → todos os alunos de um mesmo curso ficam salvos **juntos fisicamente** no mesmo nó.
- `id` é a **Clustering Key** → dentro do curso, os alunos são ordenados pelo ID.

> **💡 Regra de Ouro:** No Cassandra, a modelagem de dados é orientada pelas **consultas** que você pretende fazer, não pela estrutura dos dados em si. Se você precisa buscar alunos por curso, então `curso` deve ser a Partition Key.

**Saída esperada:**
```
📋 Tabela 'estudantes' criada com sucesso!
```

In [ ]:
# Remover tabela se existir para resetar os testes
session.execute("DROP TABLE IF EXISTS estudantes")

# Criar Tabela com chave primária composta
# ((curso)) → Partition Key (entre parênteses duplos)
# id → Clustering Key (após a vírgula)
query_tabela = """
CREATE TABLE estudantes (
    curso text,
    id int,
    nome text,
    email text,
    nota float,
    PRIMARY KEY ((curso), id)
);
"""
session.execute(query_tabela)
print("📋 Tabela 'estudantes' criada com sucesso!")

---
## 5. CRUD — Create (Inserir Dados com Prepared Statements)
No Cassandra, é uma excelente prática utilizar **Prepared Statements** (consultas preparadas). Elas trazem dois benefícios importantes:

1. **Performance:** O Cassandra compila a query apenas uma vez e reutiliza o plano de execução para todas as chamadas subsequentes.
2. **Segurança:** Previne ataques de injeção de CQL (equivalente a SQL Injection).

> **💡 Conceito-Chave:** Os `?` na query são placeholders (marcadores de posição) que serão substituídos pelos valores reais na hora da execução. Isso é diferente do Python f-string — os valores são enviados separadamente ao banco.

**Saída esperada:**
```
✍️ Estudante 'Felipe Souza' inserido no curso 'Ciência da Computação'.
✍️ Estudante 'Ana Costa' inserido no curso 'Ciência da Computação'.
✍️ Estudante 'Carlos Lima' inserido no curso 'Sistemas de Informação'.
✍️ Estudante 'Beatriz Santos' inserido no curso 'Sistemas de Informação'.
```

In [ ]:
# Preparar a query de inserção (compilada apenas uma vez pelo Cassandra)
# Os "?" são placeholders que serão substituídos pelos valores na execução
query_inserir = "INSERT INTO estudantes (curso, id, nome, email, nota) VALUES (?, ?, ?, ?, ?)"
statement_preparado = session.prepare(query_inserir)

# Dados dos estudantes (tuplas na mesma ordem dos campos da query)
estudantes = [
    ('Ciência da Computação', 1, 'Felipe Souza', 'felipe@email.com', 8.5),
    ('Ciência da Computação', 2, 'Ana Costa', 'ana.costa@email.com', 9.8),
    ('Sistemas de Informação', 3, 'Carlos Lima', 'carlos@email.com', 7.2),
    ('Sistemas de Informação', 4, 'Beatriz Santos', 'beatriz@email.com', 9.0)
]

# Executar inserção em lote sequencial
# Cada chamada reutiliza o statement preparado com valores diferentes
for est in estudantes:
    session.execute(statement_preparado, est)
    print(f"✍️ Estudante '{est[2]}' inserido no curso '{est[0]}'.")

---
## 6. CRUD — Read (Coletar e Consultar Dados)

### ⚠️ Regra de Ouro do Cassandra:
Você **só deve** consultar dados passando a **Partition Key** na cláusula `WHERE` (no nosso caso, o campo `curso`). 

Tentar buscar dados por colunas que não fazem parte da chave primária (como `email`) causará um **erro**, a menos que você force uma varredura completa com `ALLOW FILTERING` — o que é **fortemente desencorajado em produção** porque impacta negativamente a performance de todo o cluster.

```
✅ SELECT ... WHERE curso = '...'           → Consulta eficiente (usa Partition Key)
✅ SELECT ... WHERE curso = '...' AND id = 1 → Consulta eficiente (usa PK + CK)
❌ SELECT ... WHERE email = '...'            → Erro! (email não é parte da chave)
⚠️ SELECT ... WHERE email = '...' ALLOW FILTERING → Funciona, mas ineficiente!
```

**Saída esperada:**
```
📖 Consultando alunos de 'Ciência da Computação':
- ID: 1 | Nome: Felipe Souza | Nota: 8.5
- ID: 2 | Nome: Ana Costa | Nota: 9.80...
--------------------------------------------------
📖 Buscando todos os estudantes cadastrados (Select *):
(lista de todos os estudantes)
--------------------------------------------------
Tentando buscar por email...
⚠️ Erro esperado capturado: ...
```

In [ ]:
# === Consulta Eficiente (Filtrando pela Partition Key: curso) ===
# Esta é a forma CORRETA de consultar no Cassandra
print("📖 Consultando alunos de 'Ciência da Computação':")
resultados_cc = session.execute("SELECT id, nome, email, nota FROM estudantes WHERE curso = 'Ciência da Computação'")
for linha in resultados_cc:
    print(f"- ID: {linha.id} | Nome: {linha.nome} | Nota: {linha.nota}")

print("-" * 50)

# === Consulta Completa (SELECT * sem WHERE) ===
# Funciona mas retorna dados de TODAS as partições
# Em tabelas grandes, isso pode ser lento e consumir muitos recursos
print("📖 Buscando todos os estudantes cadastrados (Select *):")
todos = session.execute("SELECT * FROM estudantes")
for estudante in todos:
    print(f"Curso: {estudante.curso} | ID: {estudante.id} | Nome: {estudante.nome} | Email: {estudante.email}")

print("-" * 50)

# === Consulta Bloqueada (Tentando filtrar por campo não indexado) ===
# Isso demonstra que o Cassandra IMPEDE consultas ineficientes por padrão
try:
    print("Tentando buscar por email (campo não indexado)...")
    session.execute("SELECT * FROM estudantes WHERE email = 'felipe@email.com'")
except Exception as e:
    print(f"⚠️ Erro esperado capturado: {type(e).__name__}")
    print("O Cassandra impede essa busca porque 'email' não faz parte da chave primária.")
    
    # Usando ALLOW FILTERING como último recurso (NÃO recomendado em produção!)
    print("\n🔄 Executando com 'ALLOW FILTERING' (varredura completa — evite em produção!):")
    resultados_filtro = session.execute("SELECT * FROM estudantes WHERE email = 'felipe@email.com' ALLOW FILTERING")
    for linha in resultados_filtro:
        print(f"- Recuperado com sucesso: {linha.nome} ({linha.curso})")

---
## 7. CRUD — Update (Atualizar Dados)
No Cassandra, a escrita funciona como um **Upsert** (Update + Insert). Se você executar um `UPDATE` ou `INSERT` com a mesma chave primária composta, o Cassandra simplesmente sobrescreve os dados existentes.

> **💡 Conceito-Chave:** No `WHERE` do `UPDATE`, é obrigatório especificar a **chave primária completa** (Partition Key + Clustering Key). Não é possível atualizar múltiplos registros com uma única query genérica como no SQL.

**Saída esperada:**
```
🔄 Registro de Felipe Souza atualizado!
👤 Dados atuais: Nome: Felipe Souza | Email: felipe.novo@email.com | Nota: 9.5
```

In [ ]:
# === Atualizar a nota e e-mail do Felipe ===
# A chave primária COMPLETA deve ser especificada no WHERE:
#   - Partition Key: curso = 'Ciência da Computação'
#   - Clustering Key: id = 1
query_update = """
UPDATE estudantes 
SET nota = 9.5, email = 'felipe.novo@email.com' 
WHERE curso = 'Ciência da Computação' AND id = 1
"""
session.execute(query_update)
print("🔄 Registro de Felipe Souza atualizado!")

# Validar atualização buscando o registro
# .one() retorna uma única linha (ou None se não encontrar)
registro_atualizado = session.execute(
    "SELECT * FROM estudantes WHERE curso = 'Ciência da Computação' AND id = 1"
).one()
print(f"👤 Dados atuais: Nome: {registro_atualizado.nome} | Email: {registro_atualizado.email} | Nota: {registro_atualizado.nota}")

---
## 8. CRUD — Delete (Deletar Registros)
Para deletar, também precisamos informar a **Partition Key** (e preferencialmente a **Clustering Key**) para localizar a linha exata.

> **💡 Curiosidade:** No Cassandra, o `DELETE` não remove os dados imediatamente do disco. Ele grava um marcador especial chamado **tombstone** (lápide). Os dados são efetivamente removidos em um processo posterior chamado **compaction**.

**Saída esperada:**
```
🗑️ Carlos Lima foi removido.
📖 Alunos de Sistemas de Informação restantes:
- ID: 4 | Nome: Beatriz Santos
```

In [ ]:
# === Deletar estudante Carlos Lima ===
# Especificando a chave primária completa: Partition Key + Clustering Key
query_delete = "DELETE FROM estudantes WHERE curso = 'Sistemas de Informação' AND id = 3"
session.execute(query_delete)
print("🗑️ Carlos Lima foi removido.")

# Exibir lista final de Sistemas de Informação
print("📖 Alunos de Sistemas de Informação restantes:")
restantes_si = session.execute("SELECT * FROM estudantes WHERE curso = 'Sistemas de Informação'")
for est in restantes_si:
    print(f"- ID: {est.id} | Nome: {est.nome}")

---
## 9. Encerrando a Conexão
É uma boa prática encerrar a sessão e o cluster ao final do uso para liberar conexões e recursos de rede.

In [ ]:
# Encerrar o cluster (fecha a sessão e todas as conexões automaticamente)
cluster.shutdown()
print("🔌 Conexão com o cluster Cassandra encerrada com sucesso.")

---
## 🏁 Conclusão
Parabéns! Você concluiu os testes práticos com o Apache Cassandra:
- ✅ Criou um Keyspace e definiu fatores de replicação locais.
- ✅ Modelou uma tabela estruturada sob uma **Primary Key Composta** (Partition Key + Clustering Key).
- ✅ Entendeu a distribuição física dos dados com base na Partition Key.
- ✅ Utilizou **Prepared Statements** para inserção segura e performática.
- ✅ Entendeu as **restrições de filtragem** e como o Cassandra prioriza consultas eficientes através de chaves.

### 🚀 Próximos Passos
Para continuar se aprofundando no Cassandra, experimente:
1. **Índices Secundários (`CREATE INDEX`):** Permitir buscas por campos fora da chave primária (com custo de performance).
2. **Tabelas Materializadas (`MATERIALIZED VIEW`):** Criar visões otimizadas para padrões de consulta diferentes.
3. **Batch Statements:** Agrupar múltiplas escritas em uma única operação atômica.
4. **Níveis de Consistência:** Controlar quantos nós devem confirmar a escrita antes de retornar sucesso (`ONE`, `QUORUM`, `ALL`).
5. **Dados com TTL:** Definir expiração automática de registros com `USING TTL`.

### 📚 Referências Úteis
- [Documentação oficial do Apache Cassandra](https://cassandra.apache.org/doc/latest/)
- [Referência de CQL](https://cassandra.apache.org/doc/latest/cassandra/cql/)
- [DataStax Python Driver](https://docs.datastax.com/en/developer/python-driver/latest/)
- [Modelagem de dados no Cassandra (DataStax Academy)](https://www.datastax.com/learn/data-modeling-by-example)